# Module 4: Evaluation

Lessons from [`course/04-evaluation`](../../course/04-evaluation/) in this notebook, one section per lesson:

- [**Lesson 11** - RAG and Agent Evaluation](#lesson-11) (`11-evaluation-intro.md`)
- [**Lesson 12** - Generating RAG Answers](#lesson-12) (`12-rag-answers.md`)
- [**Lesson 13** - LLM as a Judge](#lesson-13) (`13-llm-as-judge.md`)
- [**Lesson 14** - Agent Evaluation](#lesson-14) (`14-agent-evaluation.md`)
- [**Lesson 15** - Next Steps](#lesson-15) (`15-next-steps.md`)

---

<a id="lesson-11"></a>

# Lesson 11: RAG and Agent Evaluation

Source: [`11-evaluation-intro.md`](../../course/04-evaluation/11-evaluation-intro.md) · Video: [YouTube](https://www.youtube.com/watch?v=VKHBP0QSCFo&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

So far, we evaluated retrieval. We checked whether search returns the
document that should answer the question.

That is only the first step. A complete application still needs to
produce a final answer. For RAG, this means checking the generated
answer. For agents, it also means looking at the tool calls the model
made before producing the answer.

RAG evaluation checks the whole flow together.

This includes:

- search
- prompt
- LLM

If the final answer is bad, the problem can come from any of these
steps. The search might retrieve the wrong document, the prompt might
omit important context, or the LLM might ignore the context.

In this part, we'll evaluate:

- RAG answers with an LLM judge
- Agent answers and tool-call trajectories

We won't go deep into agent evaluation frameworks here. We'll use the
agent from module 01, save the final answer, and also save the tool
calls. Then we can look at whether the answer is good and whether the
trajectory looks reasonable.

![RAG evaluation checks the search, prompt, LLM, and answer, while agent evaluation also checks tool calls and the trajectory before the judge](../../course/04-evaluation/images/11-evaluation-intro-01-rag-agent-evaluation-imagegen.png)

## LLM as a judge

For RAG and agent evaluation, we compare the generated answer with the
original answer. The generated answer won't use the same words as the
original. It's a generative model, so the phrasing will be different
even when the meaning is the same.

This is why we use another LLM to do the comparison. We show the judge
the question, the original answer, and the generated answer. Then we ask
it to decide if they are semantically equivalent.

This approach is called LLM-as-a-judge. The evaluating LLM is the
judge. It classifies each answer as good or bad and explains its
reasoning. Asking the judge to explain why it made a decision generally
produces better classifications than asking for just the verdict.

Next, we'll start with the RAG case and generate answers for the ground
truth questions.

---

<a id="lesson-12"></a>

# Lesson 12: Generating RAG Answers

Source: [`12-rag-answers.md`](../../course/04-evaluation/12-rag-answers.md) · Video: [YouTube](https://www.youtube.com/watch?v=utkcclfpj0g&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the first part of this module, we evaluated search quality. We
checked whether the right document appeared in the search results.

Now we evaluate the full RAG pipeline. For each generated question, we
run RAG and save the answer produced by the LLM. Later, we'll compare
this answer with the original FAQ answer.

This is the A->Q->A' setup:

- A = original answer in the FAQ
- Q = generated question from this answer
- A' = answer produced by our RAG system

If A' is close to A, the RAG system is doing a good job.

This is still offline evaluation. We can compare A and A' because our
questions came from FAQ records. For each question, we know which
original answer it came from.

## Loading the data

Create a new notebook for RAG evaluation.

Load the ground truth questions:

In [50]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

Load the FAQ documents and the search index:

In [51]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

Create a lookup table for the original FAQ documents:

In [52]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

We'll use this lookup table to find the original answer for each
ground truth question.

## Running RAG

Import the usual things first:

In [53]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

For this lesson, use `RAGWithUsage` from the evaluation utilities. It
subclasses `RAGBase` from module 01, so it has the same `rag` method.

It stores token usage after each LLM call. Then we can calculate the
total cost later.

It also uses the search boosts we selected in the search tuning lesson:
`question=1.0`, `answer=2.0`, and `section=0.1`.

In [54]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

For each question, `RAGBase` searches the FAQ, builds a prompt with the
retrieved context, and asks the LLM to answer. We save the answer so the
next lesson can judge it.

Run RAG for one question:

In [55]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'No, it’s not too late to join. You can start learning and submitting homework now. If you want a certificate, make sure to submit your project while submissions are still open.'

Check the cost of this call:

In [56]:
assistant.total_cost()

0.00048525

Get the original answer from the document ID:

In [57]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

Now save both answers in one record:

In [58]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Is it too late to join this course if I just found it now?',
 'answer_llm': 'No, it’s not too late to join. You can start learning and submitting homework now. If you want a certificate, make sure to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

## Processing all questions

Create a function that processes one ground truth record:

In [59]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

Test it on one record:

In [60]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Is it too late to join this course if I just found it now?',
 'answer_llm': 'No, it’s not too late to join. You can start learning and submitting homework without registering. If you want a certificate, make sure to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

Before running the full batch, reset the usage we collected while
testing:

In [61]:
assistant.reset_usage()

This calls the LLM once per ground truth question, so it can take some
time. Let's process the questions in parallel and track progress.

Import the parallel processing helper from the same utility file:

In [62]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

> **Note (not from the course):** to save money, we run RAG on a random sample of 10 ground truth questions instead of all of them. 
This is enough to try out the workflow, but too small for the scores in the next lesson to be meaningful. 
Increase `n` (e.g. 50-100) for more reliable numbers, or remove this cell to run on everything.

In [63]:
# Not from the course: use a sample of 10 questions to save money
ground_truth = df_ground_truth.sample(n=10, random_state=1).to_dict(orient="records")
len(ground_truth)

10

Run RAG for all ground truth questions:

In [64]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/10 [00:00<?, ?it/s]

`generate_rag_answer` returns one answer record for each question.

Collect the answer records:

In [65]:
answers = []

for answer_record in results:
    answers.append(answer_record)

Calculate the total cost:

In [66]:
assistant.total_cost()

0.01240125

Save the answers:

In [67]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

We generated this file for the course materials on May 29, 2026. The
run used 395 ground truth questions.

The total cost was $0.34332825, about 34 cents.

If you don't want to generate the RAG answers yourself, download the
file we prepared:

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main

wget -O data/rag-answers-new.csv ${PREFIX}/cohorts/2026/04-evaluation/data/rag-answers-new.csv
```

In the next lesson, we'll evaluate these answers with an LLM judge.

---

<a id="lesson-13"></a>

# Lesson 13: LLM as a Judge

Source: [`13-llm-as-judge.md`](../../course/04-evaluation/13-llm-as-judge.md) · Video: [YouTube](https://www.youtube.com/watch?v=BEXVULgalDM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we generated RAG answers for our ground truth
questions. Now we need to decide whether these answers are good enough.

For offline evaluation, we have three things:

- the original FAQ answer
- the question generated from that answer
- the answer generated by our RAG pipeline

An LLM judge is another LLM call that compares these three pieces. We
ask it whether the RAG answer recovers the same information as the
original answer.

It can also explain why an answer is bad.

For example:

- the retrieved document might be wrong
- the answer might miss the key point
- the model might say that it doesn't know

This approach is useful when exact text matching is too strict. The RAG
answer doesn't need to copy the FAQ answer word for word. It needs to
answer the question with the same key information.

This evaluates the full RAG flow in one pass:

- search: did we retrieve context that contains the answer?
- prompt: did we give the model enough context to answer?
- LLM: did the model produce a useful answer from that context?

If the judge marks an answer as bad, we still need to look at the
example. The judge tells us where to investigate. It doesn't replace
reading the failing cases.

## Loading the RAG answers

Start from the CSV we created in the previous lesson:

In [68]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

Each row has the generated question, the original FAQ answer, and the
answer produced by the RAG pipeline.

This is offline evaluation. We can do it because our test dataset came
from FAQ records. We know the original answer for every generated
question.

In production, we usually don't have that original answer for real user
questions. There we can still use an LLM judge. The prompt has to judge
only the question and the generated answer. In this lesson, we use the
stronger offline setup.

## A->Q->A' evaluation

We'll compare the RAG answer with the original answer from the FAQ.
This checks if the RAG pipeline is producing answers that match the
ground truth.

First, define the output format:

In [69]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

The judge returns two fields. The `score` gives us a metric we can
aggregate. The `reasoning` explains the score, which helps when we look
at bad examples.

First, write the judge instructions. This tells the judge what to
compare and how to assign the score.

In [70]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

Then define the prompt template. This is the data we pass to the judge
for each answer.

In [71]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

Import the structured-output helper:

In [72]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

Take one record:

In [73]:
rec = answers[0]

Create the judge prompt:

In [74]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

Call the judge:

In [75]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer matches the ground truth: it says no further waiting is needed, you can start learning and submitting homework while the form is open, registration is only for gauging interest, and it is not checked against a list. This preserves the core meaning and key details.', score='good')

Check the cost:

In [76]:
calc_price(usage)

{'input_cost': 0.00025425000000000003,
 'output_cost': 0.0003195,
 'total_cost': 0.00057375}

Now put the same logic into a function:

In [77]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-6-luna"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

Test it on the same record:

In [78]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer correctly says there is no need to wait, that learning and submitting homework are allowed while the form is open, and that registration is only for gauging interest and is not checked against a registered list. The added mention of a confirmation email does not contradict the original.', score='good')

## Running the judge

Run the evaluation on all answers:

In [79]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

Use the same parallel processing helper:

In [80]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/10 [00:00<?, ?it/s]

Split the results:

In [81]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

Create a dataframe:

In [82]:
df_eval = pd.DataFrame(evaluations)

Calculate the total cost:

In [83]:
calc_total_price(usages)

0.008081999999999999

Check the results:

In [84]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 9/10 = 90.00%


Look at the "bad" cases to understand what went wrong:

In [85]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
8,Do the lesson pages in the repo include the vi...,31456f4b5f,bad,The original answer explicitly says each lesso...


These rows are often the most useful part of the evaluation. They can
show that search retrieved the wrong document. They can also show that
the answer is too generic. Sometimes the RAG pipeline says that it
doesn't know even though the FAQ had the answer.

## Evaluating the judge

The judge can be wrong. It may rate an answer as good even though search
failed to retrieve the right document. In that case the judge is too
lenient. Make the instructions stricter and re-run the evaluation.

To evaluate the judge, you need to look at the results yourself. Sample
some good and bad cases, read the judge reasoning, and check whether you
agree with the verdict. You cannot use another judge to evaluate the
judge. This is manual work, but it is necessary.

A practical approach is to build a simple application with Streamlit.
Show each question, the original answer, the generated answer, and the
judge verdict side by side. Then mark each verdict as correct or
incorrect and use that feedback to adjust the judge instructions. This
is a lot of trial and error, but it makes the evaluation framework more
reliable.

## Saving the results

Save the judged answers:

In [86]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)

We generated this file for the course materials on May 29, 2026. The
run used 395 RAG answers.

The results were:

- Good: 379
- Bad: 16

The total cost was $0.251331, about 25 cents.

If you don't want to run the judge yourself, download the file we
prepared:

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
wget -O data/rag-evaluations-new.csv ${PREFIX}/cohorts/2026/04-evaluation/data/rag-evaluations-new.csv
```

We now have an answer-quality score for the RAG pipeline. In the next
lesson, we'll apply the same idea to an agent and also capture the tool
calls it made.

---

<a id="lesson-14"></a>

# Lesson 14: Agent Evaluation

Source: [`14-agent-evaluation.md`](../../course/04-evaluation/14-agent-evaluation.md) · Video: [YouTube](https://www.youtube.com/watch?v=2SW86BehVdI&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

For RAG, we used the A->Q->A' setup:

- A = original answer in the FAQ
- Q = generated question from this answer
- A' = answer produced by our RAG system

For agents, we use the same setup. A' comes from an agent instead of a
fixed RAG pipeline.

We also save the trajectory. Here, the trajectory means only the tool
calls the agent made before producing the final answer.

![Agent evaluation stores the question, original answer, generated answer, tool calls, and trajectory in an evaluation record for a judge](../../course/04-evaluation/images/14-agent-evaluation-01-agent-evaluation-record-imagegen.png)

## Loading the data

Use the same ground truth questions:

In [87]:
import json
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [88]:
# Not from the course: use a sample of 10 questions to save money
ground_truth = df_ground_truth.sample(n=10, random_state=1).to_dict(orient="records")
len(ground_truth)

10

Load the FAQ documents and the search index:

In [89]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

Create a lookup table:

In [90]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

## Running the agent

Reuse the ToyAIKit agent from module 01. It handles the agent loop and
stores the full message history.

First, set up the model clients:

In [91]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

Define the search tool:

In [92]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

Create the runner:

In [93]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-6-luna")
)

The result contains:

- `last_message`: the final response
- `all_messages`: the full message history
- `cost`: the cost of all LLM calls in this run

Run it for one ground truth question:

In [94]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

Look at the full message history:

In [95]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='If I already registered, should I wait for anything else before learning and turning in assignments?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"After registering, do students need to wait for anything else before starting learning or submitting assignments?"}', call_id='call_0623a3b9869d45c8811e571c7cdfd795', name='search', type='function_call', id='fc_0d8ecee7f59302a1006b082151064087d2aa62b975f293b845', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_0623a3b9869d45c8811e571c7cdfd795',
  'output': '[\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have regist

For this lesson, the trajectory is only the tool calls. We don't need
to send the full message history to the judge.

Extract the function name and arguments:

In [96]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

For this example:

In [97]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"After registering, do students need to wait for anything else before starting learning or submitting assignments?"}'}]

You should see something like this:

```python
[
    {
        "name": "search",
        "arguments": "{\"query\":\"own pace certificate at the end self-paced course certificate\"}"
    }
]
```

Get the original answer:

In [98]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

Save the A->Q->A' record and the trajectory:

In [99]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": json.dumps(tool_calls),
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'If I already registered, should I wait for anything else before learning and turning in assignments?',
 'answer_agent': 'No. Once you’ve registered, you can start learning and submitting assignments right away—there’s no confirmation email or other approval to wait for. Registration is just to gauge interest, and homework isn’t checked against a registration list. You can also start even if you haven’t registered, as long as the submission form is open.',
 'answer_orig': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
 'tool_calls': '[{"name": "search", "arguments": "{\\"query\\":\\"After registering, do students need to wait for anything else before starting learning or submitting assignments?\\"}"}]',
 'cost': Decimal('0.0001376'),
 'document': '977bf7786c'}

The `answer_agent` field is what we evaluate with the LLM judge. The
`tool_calls` field lets the judge see how the agent got there. 
Wrap `tool_calls` with `json.dumps` to properly escape special characters,
such as double quotes.

## Processing multiple questions

Create a function that processes one ground truth record:

In [100]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": json.dumps(tool_calls),
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

Run it for a small sample in parallel:

In [101]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/10 [00:00<?, ?it/s]

Turn it into a dataframe:

In [102]:
df_agent = pd.DataFrame(agent_answers)

Calculate the total cost:

In [103]:
df_agent["cost"].sum()

Decimal('0.0022287')

Save the results:

In [104]:
df_agent.to_csv("data/agent-answers.csv", index=False)

Now we have the same A->Q->A' data as before, plus the tool calls for
each agent run.

We generated this file for the course materials on May 29, 2026. The
run used 50 ground truth questions. ToyAIKit tracks the agent cost for
each run, so we can sum the `cost` column directly.

The total agent cost was $0.06993300, about 7 cents.

If you don't want to run the agent yourself, download the file we
prepared:

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
wget -O data/agent-answers.csv ${PREFIX}/cohorts/2026/04-evaluation/data/agent-answers.csv
```

Then load it:

In [105]:
df_agent = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

Our judge can look at both:

- whether `answer_agent` matches `answer_orig`
- whether the tool calls look reasonable for the question

This lets us evaluate the final answer and the agent behavior in one
place.

## Judging answers and trajectories

A good trajectory is not just "many tool calls". A good trajectory uses
the available tools in a way that helps answer the question.

For our search agent, a good trajectory has these properties:

- The search query is relevant to the user question
- The query includes the important keywords from the question
- The agent avoids duplicate searches with the same arguments
- If it searches more than once, the next query is a useful refinement
- It usually uses 1 search call
- 2-3 calls can be okay for harder questions
- More than 3 search calls needs a clear reason
- The tool calls support the final answer
- The agent does not stop too early or keep searching without a reason

Now define a judge output type with two scores:

In [106]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

The judge instructions:

In [107]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

Define the judge function:

In [108]:
from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gpt-6-luna"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

Test it on one agent result:

In [109]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning="The answer correctly conveys that registration does not gate access and that the student can begin learning and submit homework while the form is open. The added statement about not needing a confirmation email is not explicit in the original, but it is consistent with the original's instruction that no further step is needed.", answer_score='good', trajectory_reasoning='The single search query directly addresses registration, starting to learn, and submitting assignments. It is relevant and includes the key concepts without unnecessary duplication.', trajectory_score='good')

When the answer is bad, the trajectory score tells us whether the
problem started with tool use. If the answer is bad but the trajectory
is good, the model may have used the retrieved context poorly.
If both are bad, the agent likely searched for the wrong thing. It may
also have stopped too early.

## Running the agent judge

Run the judge for all agent answers:

In [110]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

Use the same parallel helper:

In [111]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/10 [00:00<?, ?it/s]

Split the results:

In [112]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

Create a dataframe:

In [113]:
df_agent_eval = pd.DataFrame(agent_evaluations)

Calculate the judge cost from the token usage:

In [114]:
calc_total_price(usages)

0.01150425

Check the answer scores:

In [115]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    10
Name: count, dtype: int64

Check the trajectory scores:

In [116]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    10
Name: count, dtype: int64

Save the judge results:

In [117]:
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)

We generated this file for the course materials on May 29, 2026. The
run judged 50 agent answers.

The answer scores were:

- Good: 45
- Bad: 5

The trajectory scores were:

- Good: 49
- Bad: 1

The judge token usage was:

- Input tokens: 29,228
- Output tokens: 6,984
- Cost with the prices above: $0.053349, about 5 cents

If you don't want to run the judge yourself, download the file we
prepared:

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main

wget -O data/agent-evaluations.csv ${PREFIX}/cohorts/2026/04-evaluation/data/agent-evaluations.csv
```

---

<a id="lesson-15"></a>

# Lesson 15: Next Steps

Source: [`15-next-steps.md`](../../course/04-evaluation/15-next-steps.md) · Video: [YouTube](https://www.youtube.com/watch?v=TlKPBjItUw8&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we covered evaluation at three levels:

1. Search evaluation: Hit Rate and MRR to measure retrieval quality
2. RAG evaluation: LLM-as-a-judge for answer quality
3. Agent evaluation: final answers plus tool-call trajectories

Evaluation is not a one-time activity. As you tune search parameters,
switch models, or modify prompts, re-run evaluation. Make sure the
system is getting better, not worse.

![Evaluation feedback loop from synthetic questions through evaluation and deployment to real queries, human labels, and an improved test set](../../course/04-evaluation/images/15-next-steps-01-evaluation-feedback-loop-imagegen.png)

Evaluation is the most important part of building AI systems. It is also
the most time-consuming. Only after evaluation can you be confident
that your system works. Validate every change against your evaluation
framework before going to production. This applies to prompt updates,
model swaps, and agent modifications.

## From synthetic data to real data

The evaluation workflow in practice:

1. Start with synthetic data. Use an LLM to generate questions from
   your FAQ or documentation. This gives you a baseline without needing
   real users.
2. Tune the data generation. If the metrics look suspiciously good,
   the synthetic questions may be too close to the source text. Adjust
   the generation prompt to produce more realistic questions.
3. Deploy and collect real data. Once the system is in production, start
   collecting actual user queries and feedback.
4. Label real data. Have humans label whether the retrieved documents
   and generated answers are correct. This produces the most reliable
   ground truth.
5. Tune synthetic generation to match real data. Use the patterns from
   real queries to improve your synthetic data generator. The closer
   your synthetic data is to real data, the more useful the metrics
   become.

Nothing beats manual evaluation. Try the system yourself, think about
edge cases, and collect examples of where it fails. This is especially
important in the early stages when you don't have automated evaluation
set up yet.

## Evaluation frameworks

For production systems, consider using evaluation frameworks that make
it easier to manage test datasets, run evaluations, and track results:

- Ragas: a framework for evaluating RAG systems with metrics like
  faithfulness, answer relevance, and context precision
- DeepEval: provides built-in metrics for RAG evaluation including
  hallucination detection and answer relevance
- TruLens: instruments your LLM app and tracks quality metrics

These frameworks implement many of the concepts we covered here and
add visualizations and experiment tracking.

## Monitoring

Online evaluation (monitoring) is what you do after deploying your
system.

Key approaches:

- User feedback: thumbs up/down buttons to collect signal
- Logging: record queries, retrieved documents, and answers
- Dashboards: track metrics over time to spot degradation
- Alerts: get notified when metrics drop below a threshold

Monitoring is covered in more detail in module 05.

## To learn more

See also:

- Cohorts and materials:
  - 2024 cohort evaluation module (uses Elasticsearch):
    [2024/04-monitoring](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/cohorts/2024/04-monitoring)
  - 2025 cohort evaluation module:
    [2025/03-evaluation](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/cohorts/2025/03-evaluation)